## 1. Hướng dẫn Add Dataset vào Kaggle
Trong Kaggle, bạn KHÔNG CẦN dùng lệnh `!git clone` vì nó rất chậm và hay lỗi.
1. Nhìn sang cột bên phải của giao diện Kaggle, bấm nút **Add Input** (hoặc Add Data).
2. Gõ tìm kiếm: **PlantDoc Classification** (hoặc Plant Village) và bấm phím **+ (Add)**.
3. Sửa đường dẫn biến `DATASET_PATH` ở Ô Code bên dưới thành đường dẫn thư mục `train` của dataset bạn vừa thêm (thường nằm trong `/kaggle/input/tên-dataset/train`).

In [ ]:
import os

# Script tự động tìm thư mục dữ liệu của bạn
DATASET_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    # PyTorch ImageFolder yêu cầu một thư mục chứa nhiều thư mục con (mỗi thư mục con là 1 loại bệnh).
    # Chúng ta tìm thư mục nào có từ 10 thư mục con trở lên.
    if len(dirs) >= 10:
        DATASET_PATH = root
        break

if DATASET_PATH:
    print(f'🎉 TUYỆT VỜI! Đã tự động tìm thấy dữ liệu tại: {DATASET_PATH}')
else:
    print('❌ LỖI: Kaggle có thể đang mất thời gian để tải dữ liệu vào phiên làm việc. Bạn hãy đợi khoảng 1 phút rồi bấm chạy lại ô Code này nhé!')


## 2. Import Thư viện PyTorch

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Sử dụng thiết bị: {device}')

## 3. Load Dữ liệu

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Lưu tên các loại bệnh ra file json để Backend sử dụng
class_indices = {v: k for k, v in train_dataset.class_to_idx.items()}
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f)
print(f'Số lượng loại bệnh: {len(class_indices)}')


## 4. Xây dựng Model (Sử dụng Transfer Learning ResNet18)

In [ ]:
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_indices))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 5. Huấn luyện Model

In [ ]:
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}')

## 6. Lưu file Model

In [ ]:
torch.save(model.state_dict(), 'best_model.pt')
print('Đã lưu model thành best_model.pt! Vui lòng tải file này và class_indices.json về máy.')